In [0]:
silver_candidate = spark.sql("""
SELECT
    CAST(GlobalEventID AS BIGINT) AS GlobalEventID,

    try_to_date(SQLDATE, 'yyyyMMdd') AS EventDate,
    CAST(MonthYear AS INT) AS MonthYear,
    CAST(Year AS INT) AS Year,
    try_cast(FractionDate AS DOUBLE) AS FractionDate,

    Actor1Code,
    Actor1Name,
    Actor1CountryCode,
    Actor1KnownGroupCode,
    Actor1EthnicCode,
    Actor1Religion1Code,
    Actor1Religion2Code,
    Actor1Type1Code,
    Actor1Type2Code,
    Actor1Type3Code,

    Actor2Code,
    Actor2Name,
    Actor2CountryCode,
    Actor2KnownGroupCode,
    Actor2EthnicCode,
    Actor2Religion1Code,
    Actor2Religion2Code,
    Actor2Type1Code,
    Actor2Type2Code,
    Actor2Type3Code,

    try_cast(IsRootEvent AS INT) AS IsRootEvent,

    EventCode,
    EventBaseCode,
    EventRootCode,

    try_cast(QuadClass AS INT) AS QuadClass,
    CAST(GoldsteinScale AS DOUBLE) AS GoldsteinScale,
    try_cast(NumMentions AS INT) AS NumMentions,
    try_cast(NumSources AS INT) AS NumSources,
    try_cast(NumArticles AS INT) AS NumArticles,
    try_cast(AvgTone AS DOUBLE) AS AvgTone,

    try_cast(Actor1Geo_Type AS INT) AS Actor1Geo_Type,
    Actor1Geo_FullName,
    Actor1Geo_CountryCode,
    Actor1Geo_ADM1Code,
    try_cast(Actor1Geo_Lat AS DOUBLE) AS Actor1Geo_Lat,
    try_cast(Actor1Geo_Long AS DOUBLE) AS Actor1Geo_Long,
    Actor1Geo_FeatureID,

    try_cast(Actor2Geo_Type AS INT) AS Actor2Geo_Type,
    Actor2Geo_FullName,
    Actor2Geo_CountryCode,
    Actor2Geo_ADM1Code,
    try_cast(Actor2Geo_Lat AS DOUBLE) AS Actor2Geo_Lat,
    try_cast(Actor2Geo_Long AS DOUBLE) AS Actor2Geo_Long,
    Actor2Geo_FeatureID,

    try_cast(ActionGeo_Type AS INT) AS ActionGeo_Type,
    ActionGeo_FullName,
    ActionGeo_CountryCode,
    ActionGeo_ADM1Code,
    try_cast(ActionGeo_Lat AS DOUBLE) AS ActionGeo_Lat,
    try_cast(ActionGeo_Long AS DOUBLE) AS ActionGeo_Long,
    ActionGeo_FeatureID,

    DATEADDED,
    SOURCEURL

FROM workspace.bronze.events_raw
""")

display(silver_candidate.limit(5))

In [0]:
silver_candidate.printSchema()

In [0]:
quality_check = silver_candidate.selectExpr(
    "COUNT(*) AS total_records",

    "SUM(CASE WHEN GlobalEventID IS NULL THEN 1 ELSE 0 END) AS invalid_event_id",

    "SUM(CASE WHEN EventDate IS NULL THEN 1 ELSE 0 END) AS invalid_event_date",

    """SUM(
        CASE
            WHEN Actor1Geo_Lat IS NOT NULL
             AND (Actor1Geo_Lat < -90 OR Actor1Geo_Lat > 90)
            THEN 1 ELSE 0
        END
    ) AS invalid_actor1_lat""",

    """SUM(
        CASE
            WHEN Actor1Geo_Long IS NOT NULL
             AND (Actor1Geo_Long < -180 OR Actor1Geo_Long > 180)
            THEN 1 ELSE 0
        END
    ) AS invalid_actor1_long""",

    """SUM(
        CASE
            WHEN Actor2Geo_Lat IS NOT NULL
             AND (Actor2Geo_Lat < -90 OR Actor2Geo_Lat > 90)
            THEN 1 ELSE 0
        END
    ) AS invalid_actor2_lat""",

    """SUM(
        CASE
            WHEN Actor2Geo_Long IS NOT NULL
             AND (Actor2Geo_Long < -180 OR Actor2Geo_Long > 180)
            THEN 1 ELSE 0
        END
    ) AS invalid_actor2_long""",

    """SUM(
        CASE
            WHEN ActionGeo_Lat IS NOT NULL
             AND (ActionGeo_Lat < -90 OR ActionGeo_Lat > 90)
            THEN 1 ELSE 0
        END
    ) AS invalid_action_lat""",

    """SUM(
        CASE
            WHEN ActionGeo_Long IS NOT NULL
             AND (ActionGeo_Long < -180 OR ActionGeo_Long > 180)
            THEN 1 ELSE 0
        END
    ) AS invalid_action_long""",

    """SUM(
        CASE
            WHEN QuadClass IS NOT NULL
             AND QuadClass NOT BETWEEN 1 AND 4
            THEN 1 ELSE 0
        END
    ) AS invalid_quad_class"""
)

display(quality_check)

In [0]:
display(silver_candidate.limit(5))

In [0]:
from pyspark.sql import functions as F

numeric_checks = {
    "FractionDate": "double",
    "IsRootEvent": "int",
    "QuadClass": "int",
    "GoldsteinScale": "double",
    "NumMentions": "int",
    "NumSources": "int",
    "NumArticles": "int",
    "AvgTone": "double",
    "Actor1Geo_Type": "int",
    "Actor1Geo_Lat": "double",
    "Actor1Geo_Long": "double",
    "Actor2Geo_Type": "int",
    "Actor2Geo_Lat": "double",
    "Actor2Geo_Long": "double",
    "ActionGeo_Type": "int",
    "ActionGeo_Lat": "double",
    "ActionGeo_Long": "double"
}

for column, dtype in numeric_checks.items():
    bad = spark.sql(f"""
        SELECT COUNT(*) AS bad_count
        FROM workspace.bronze.events_raw
        WHERE {column} IS NOT NULL
          AND try_cast({column} AS {dtype}) IS NULL
    """).collect()[0]["bad_count"]

    if bad > 0:
        print(f"{column}: {bad} malformed values")

In [0]:
for column, dtype in numeric_checks.items():
    bad_rows = spark.sql(f"""
        SELECT DISTINCT {column}
        FROM workspace.bronze.events_raw
        WHERE {column} IS NOT NULL
          AND try_cast({column} AS {dtype}) IS NULL
        LIMIT 10
    """).collect()

    if bad_rows:
        print(f"\n--- {column} ---")
        for row in bad_rows:
            print(row[0])

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.bronze.events_raw
        WHERE ActionGeo_Long   = '42#.5'
    """)
)

In [0]:
silver_checked = silver_candidate.withColumn(
    "is_valid_event_id",
    F.col("GlobalEventID").isNotNull()
).withColumn(
    "is_valid_date",
    F.col("EventDate").isNotNull()
).withColumn(
    "is_valid_quad_class",
    F.col("QuadClass").isNull() | F.col("QuadClass").between(1, 4)
).withColumn(
    "is_valid_actor1_lat",
    F.col("Actor1Geo_Lat").isNull() |
    F.col("Actor1Geo_Lat").between(-90, 90)
).withColumn(
    "is_valid_actor1_long",
    F.col("Actor1Geo_Long").isNull() |
    F.col("Actor1Geo_Long").between(-180, 180)
).withColumn(
    "is_valid_actor2_lat",
    F.col("Actor2Geo_Lat").isNull() |
    F.col("Actor2Geo_Lat").between(-90, 90)
).withColumn(
    "is_valid_actor2_long",
    F.col("Actor2Geo_Long").isNull() |
    F.col("Actor2Geo_Long").between(-180, 180)
).withColumn(
    "is_valid_action_lat",
    F.col("ActionGeo_Lat").isNull() |
    F.col("ActionGeo_Lat").between(-90, 90)
).withColumn(
    "is_valid_action_long",
    F.col("ActionGeo_Long").isNull() |
    F.col("ActionGeo_Long").between(-180, 180)
)

In [0]:
silver_checked = silver_checked.withColumn(
    "is_valid_record",
    (
        F.col("is_valid_event_id") &
        F.col("is_valid_date") &
        F.col("is_valid_quad_class") &
        F.col("is_valid_actor1_lat") &
        F.col("is_valid_actor1_long") &
        F.col("is_valid_actor2_lat") &
        F.col("is_valid_actor2_long") &
        F.col("is_valid_action_lat") &
        F.col("is_valid_action_long")
    )
)

In [0]:
display(
    silver_checked.groupBy("is_valid_record")
    .count()
    .orderBy("is_valid_record")
)

In [0]:
total = silver_checked.count()

invalid = (
    silver_checked
    .filter(~F.col("is_valid_record"))
    .count()
)

print(f"Total records: {total:,}")
print(f"Invalid records: {invalid:,}")
print(f"Invalid %: {(invalid / total) * 100:.2f}%")

Let's recreate silver_checked with a separate malformed-value flag to fix the ActionGeo longitude rule

In [0]:
silver_checked = (
    silver_candidate
    .withColumn(
        "is_valid_event_id",
        F.col("GlobalEventID").isNotNull()
    )
    .withColumn(
        "is_valid_date",
        F.col("EventDate").isNotNull()
    )
    .withColumn(
        "is_valid_quad_class",
        F.col("QuadClass").isNull() |
        F.col("QuadClass").between(1, 4)
    )
)


In [0]:
raw_action_long = (
    spark.table("workspace.bronze.events_raw")
    .select("GlobalEventID", "ActionGeo_Long")
)

In [0]:
silver_checked = (
    silver_candidate
    .join(raw_action_long, on="GlobalEventID", how="left")
)

In [0]:
raw_action_long = (
    spark.table("workspace.bronze.events_raw")
    .select(
        "GlobalEventID",
        F.col("ActionGeo_Long").alias("raw_ActionGeo_Long")
    )
)

silver_checked = (
    silver_candidate
    .join(raw_action_long, on="GlobalEventID", how="left")
)

In [0]:
from pyspark.sql import functions as F

silver_checked = (
    silver_candidate

    # Required-field checks
    .withColumn(
        "is_valid_event_id",
        F.col("GlobalEventID").isNotNull()
    )

    .withColumn(
        "is_valid_date",
        F.col("EventDate").isNotNull()
    )

    # CAMEO QuadClass must be 1-4.
    # NULL is allowed here because missing source data is different
    # from an invalid value.
    .withColumn(
        "is_valid_quad_class",
        F.col("QuadClass").isNull() |
        F.col("QuadClass").between(1, 4)
    )

    # Geographic range checks
    .withColumn(
        "is_valid_actor1_lat",
        F.col("Actor1Geo_Lat").isNull() |
        F.col("Actor1Geo_Lat").between(-90, 90)
    )

    .withColumn(
        "is_valid_actor1_long",
        F.col("Actor1Geo_Long").isNull() |
        F.col("Actor1Geo_Long").between(-180, 180)
    )

    .withColumn(
        "is_valid_actor2_lat",
        F.col("Actor2Geo_Lat").isNull() |
        F.col("Actor2Geo_Lat").between(-90, 90)
    )

    .withColumn(
        "is_valid_actor2_long",
        F.col("Actor2Geo_Long").isNull() |
        F.col("Actor2Geo_Long").between(-180, 180)
    )

    .withColumn(
        "is_valid_action_lat",
        F.col("ActionGeo_Lat").isNull() |
        F.col("ActionGeo_Lat").between(-90, 90)
    )

    # IMPORTANT:
    # Because try_cast("42#.5" AS DOUBLE) became NULL,
    # we need to distinguish a legitimate source NULL
    # from a malformed source value.
    .withColumn(
        "is_valid_action_long",
        F.col("ActionGeo_Long").isNull() |
        F.col("ActionGeo_Long").between(-180, 180)
    )
)

In [0]:
raw_action_long = (
    spark.table("workspace.bronze.events_raw")
    .select(
        "GlobalEventID",
        F.col("ActionGeo_Long").alias("raw_ActionGeo_Long")
    )
)

silver_checked = (
    silver_candidate
    .join(raw_action_long, on="GlobalEventID", how="left")
)

In [0]:
silver_checked = (
    silver_checked

    .withColumn(
        "is_valid_event_id",
        F.col("GlobalEventID").isNotNull()
    )

    .withColumn(
        "is_valid_date",
        F.col("EventDate").isNotNull()
    )

    .withColumn(
        "is_valid_quad_class",
        F.col("QuadClass").isNull() |
        F.col("QuadClass").between(1, 4)
    )

    .withColumn(
        "is_valid_actor1_lat",
        F.col("Actor1Geo_Lat").isNull() |
        F.col("Actor1Geo_Lat").between(-90, 90)
    )

    .withColumn(
        "is_valid_actor1_long",
        F.col("Actor1Geo_Long").isNull() |
        F.col("Actor1Geo_Long").between(-180, 180)
    )

    .withColumn(
        "is_valid_actor2_lat",
        F.col("Actor2Geo_Lat").isNull() |
        F.col("Actor2Geo_Lat").between(-90, 90)
    )

    .withColumn(
        "is_valid_actor2_long",
        F.col("Actor2Geo_Long").isNull() |
        F.col("Actor2Geo_Long").between(-180, 180)
    )

    .withColumn(
        "is_valid_action_lat",
        F.col("ActionGeo_Lat").isNull() |
        F.col("ActionGeo_Lat").between(-90, 90)
    )

    .withColumn(
        "is_malformed_action_long",
        F.col("raw_ActionGeo_Long").isNotNull() &
        F.col("ActionGeo_Long").isNull()
    )

    .withColumn(
        "is_valid_action_long",
        F.col("raw_ActionGeo_Long").isNull() |
        (
            F.col("ActionGeo_Long").isNotNull() &
            F.col("ActionGeo_Long").between(-180, 180)
        )
    )
)

In [0]:
silver_checked = silver_checked.withColumn(
    "is_valid_record",
    F.col("is_valid_event_id")
    & F.col("is_valid_date")
    & F.col("is_valid_quad_class")
    & F.col("is_valid_actor1_lat")
    & F.col("is_valid_actor1_long")
    & F.col("is_valid_actor2_lat")
    & F.col("is_valid_actor2_long")
    & F.col("is_valid_action_lat")
    & F.col("is_valid_action_long")
)

In [0]:
display(
    silver_checked
    .filter(F.col("is_malformed_action_long"))
    .select(
        "GlobalEventID",
        "raw_ActionGeo_Long",
        "ActionGeo_Long",
        "is_malformed_action_long",
        "is_valid_action_long",
        "is_valid_record"
    )
)

In [0]:
total = silver_checked.count()

invalid = (
    silver_checked
    .filter(~F.col("is_valid_record"))
    .count()
)

print(f"Total records: {total:,}")
print(f"Invalid records: {invalid:,}")
print(f"Invalid %: {(invalid / total) * 100:.4f}%")

In [0]:
silver_events = (
    silver_checked
    .filter(F.col("is_valid_record"))
    .drop(
        "raw_ActionGeo_Long",
        "is_valid_event_id",
        "is_valid_date",
        "is_valid_quad_class",
        "is_valid_actor1_lat",
        "is_valid_actor1_long",
        "is_valid_actor2_lat",
        "is_valid_actor2_long",
        "is_valid_action_lat",
        "is_valid_action_long",
        "is_malformed_action_long",
        "is_valid_record"
    )
)

In [0]:
print("Silver records:", silver_events.count())

In [0]:
rejected_events = (
    spark.table("workspace.bronze.events_raw")
    .join(
        silver_checked
        .filter(~F.col("is_valid_record"))
        .select("GlobalEventID"),
        on="GlobalEventID",
        how="inner"
    )
    .withColumn(
        "rejection_reason",
        F.when(
            F.col("ActionGeo_Long").isNotNull() &
            F.col("ActionGeo_Long").rlike("[^0-9.+-]"),
            F.lit("MALFORMED_ACTION_LONG")
        ).otherwise(
            F.lit("DATA_QUALITY_RULE_FAILED")
        )
    )
)

In [0]:
display(
    rejected_events.select(
        "GlobalEventID",
        "ActionGeo_Long",
        "rejection_reason",
        "SOURCEURL"
    )
)

In [0]:
(
    silver_events.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.events")
)

In [0]:
(
    rejected_events.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.events_rejected")
)

In [0]:
display(
    spark.sql("""
        SELECT COUNT(*) AS silver_count
        FROM workspace.silver.events
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS rejected_count
        FROM workspace.silver.events_rejected
    """)
)

In [0]:
spark.sql("""
DESCRIBE workspace.silver.events
""").show(100, truncate=False)

**DeDuplication**

In [0]:
duplicates = spark.sql("""
SELECT
    GlobalEventID,
    COUNT(*) AS occurrences
FROM workspace.bronze.events_raw
GROUP BY GlobalEventID
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
""")

display(duplicates)